In [1]:
"""
CineMatch — XSimGCL Collaborative Filtering via RecBole
========================================================
Trains an XSimGCL (Cross-batch Simulated Graph Contrastive Learning) model
on the MovieLens 32M dataset using RecBole. Exports user/item embeddings
for downstream late fusion.

Features:
  - Automatic conversion of MovieLens CSV → RecBole .inter format
  - XSimGCL config with tuned hyperparameters for long-tail items
  - Exports user_embeddings.npy and item_embeddings.npy
  - ID mapping files for MovieLens ↔ RecBole translation
  - New-user warm-start via item embedding aggregation
  - Colab / HPC / local path auto-detection
"""

'\nCineMatch — XSimGCL Collaborative Filtering via RecBole\n========================================================\nTrains an XSimGCL (Cross-batch Simulated Graph Contrastive Learning) model\non the MovieLens 32M dataset using RecBole. Exports user/item embeddings\nfor downstream late fusion.\n\nFeatures:\n  - Automatic conversion of MovieLens CSV → RecBole .inter format\n  - XSimGCL config with tuned hyperparameters for long-tail items\n  - Exports user_embeddings.npy and item_embeddings.npy\n  - ID mapping files for MovieLens ↔ RecBole translation\n  - New-user warm-start via item embedding aggregation\n  - Colab / HPC / local path auto-detection\n'

In [2]:
# # !pip install recbole==1.1.1
# # !pip install torch-geometric
# # !git clone https://github.com/RUCAIBox/RecBole-GNN.git
# import sys
# sys.path.insert(0, "/content/RecBole-GNN")

In [3]:
# !pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster  --y
# !pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install git+https://github.com/pyg-team/pytorch_geometric.git

In [4]:
import os
import warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

import torch
if not hasattr(torch, "_original_load"):
    torch._original_load = torch.load
    def safe_load(f, map_location=None, pickle_module=None, **kwargs):
        kwargs.setdefault("weights_only", False)
        return torch._original_load(f, map_location=map_location, **kwargs)
    torch.load = safe_load
    
import sys
import time
import json
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from pathlib import Path
from __future__ import annotations


# CONFIG

# XSimGCL Hyperparameters
EMBEDDING_SIZE = 512      # dimensionality of user/item embeddings
N_LAYERS       = 3         # number of GCN layers
CL_RATE        = 0.5       # contrastive learning loss weight
NOISE_EPS      = 0.1       # noise perturbation epsilon for SimGCL
REG_WEIGHT     = 1e-4      # L2 regularization

# Training
LEARNING_RATE  = 1e-3
TRAIN_BATCH    = 294912
EPOCHS         = 50     # max epochs
EARLY_STOP     = 5        # patience
EVAL_BATCH     = 40000000

# Data
RATING_THRESHOLD = 3.5     # implicit positive threshold
TEMPORAL_SPLIT   = True    # True = temporal split, False = RecBole random
SPLIT_RATIO      = [0.8, 0.1, 0.1]  # train/val/test

# Demographics (for website users)
AGE_BUCKETS = ["18-24", "25-34", "35-44", "45-54", "55+"]
GENDER_OPTIONS = ["M", "F", "undisclosed"]
REGION_OPTIONS = [
    "USA",
    "Canada", "UK", "Europe", "Latin-America",
    "Asia", "Middle-East", "Africa", "Other",
]
DEMO_BLEND_WEIGHT = 0.3    # weight for demographic prior when blending with few interactions
MIN_CLUSTER_SIZE  = 5      # minimum users per cluster to form a centroid


@dataclass
class UserProfile:
    """Demographic profile collected from website signup."""
    age_group: str = "undisclosed"   # one of AGE_BUCKETS or "undisclosed"
    gender: str    = "undisclosed"   # one of GENDER_OPTIONS
    region: str    = "Other"         # one of REGION_OPTIONS

    def cluster_keys(self) -> list[tuple]:
        """Return lookup keys from most to least specific for fallback matching."""
        return [
            (self.age_group, self.gender, self.region),   # exact
            (self.age_group, self.gender, "*"),            # drop region
            (self.age_group, "*", "*"),                    # age only
            ("*", self.gender, "*"),                        # gender only
            ("*", "*", self.region),                        # region only
        ]

In [5]:
# PATHS
def detect_paths() -> dict:
    try:
        from google.colab import drive     # type: ignore
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cinematch/Data")
        print("Runtime: Colab")
    except ImportError:
        hpc = Path("/blue/egn6933/nagabhairava.r")
        if hpc.exists():
            base = hpc
            print("Runtime: HPC")
        else:
            here = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
            for candidate in [here, *here.parents]:
                if (candidate / "Data").exists() and (candidate / "src").exists():
                    base = candidate / "Data"
                    break
            else:
                base = Path.cwd() / "Data"
            print("Runtime: Local")

    model_dir = base / "outputs" / "xsimgcl"
    if not model_dir.is_absolute():
        model_dir = model_dir.resolve()
    model_dir.mkdir(parents=True, exist_ok=True)

    # RecBole needs a dataset dir with <dataset_name>/<dataset_name>.inter
    recbole_data = model_dir / "dataset" / "cinematch"
    recbole_data.mkdir(parents=True, exist_ok=True)

    # ML-32M paths
    ml_dir = base / "ml-32m"
    if not ml_dir.exists():
        ml_dir = base / "Data" / "ml-32m"

    return {
        "base":            base,
        "ratings_csv":     ml_dir / "ratings.csv",
        "movies_csv":      ml_dir / "movies.csv",
        "links_csv":       ml_dir / "links.csv",
        "model_dir":       model_dir,
        "recbole_data":    recbole_data,
        "inter_file":      recbole_data / "cinematch.inter",
        "user_emb":        model_dir / "user_embeddings.npy",
        "item_emb":        model_dir / "item_embeddings.npy",
        "user_id_map":     model_dir / "user_id_map.json",
        "item_id_map":     model_dir / "item_id_map.json",
        "config_yaml":     model_dir / "xsimgcl_config.yaml",
        "train_manifest":  model_dir / "train_manifest.json",
        # Demographic files (populated once website users accumulate)
        "demo_profiles":     model_dir / "user_demographics.csv",
        "demo_clusters":     model_dir / "demographic_clusters.npy",
        "demo_cluster_map":  model_dir / "demographic_cluster_map.json",
    }

In [6]:
# DATA CONVERSION
def convert_movielens_to_recbole(paths: dict) -> tuple[dict, dict]:
    print("Converting MovieLens to RecBole .inter format")
    print(f"{'─'*60}")

    assert paths["ratings_csv"].exists(), f"Missing: {paths['ratings_csv']}"

    # Load ratings
    dtypes = {"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int32"}
    ratings = pd.read_csv(paths["ratings_csv"], dtype=dtypes)
    print(f"  Loaded {len(ratings):,} ratings")

    # Convert to implicit: keep only positive interactions
    ratings = ratings[ratings["rating"] >= RATING_THRESHOLD].copy()
    print(f"  After threshold ({RATING_THRESHOLD}): {len(ratings):,} positive interactions")

    # Build contiguous ID mappings for RecBole efficiency
    unique_users = sorted(ratings["userId"].unique())
    unique_items = sorted(ratings["movieId"].unique())

    user_id_map = {orig: idx for idx, orig in enumerate(unique_users)}
    item_id_map = {orig: idx for idx, orig in enumerate(unique_items)}

    # Map to contiguous IDs
    ratings["user_id"] = ratings["userId"].map(user_id_map)
    ratings["item_id"] = ratings["movieId"].map(item_id_map)

    print(f"  Users: {len(unique_users):,}  |  Items: {len(unique_items):,}")
    print(f"  Density: {len(ratings) / (len(unique_users) * len(unique_items)) * 100:.4f}%")

    # Sort by timestamp for temporal split
    if TEMPORAL_SPLIT:
        ratings = ratings.sort_values("timestamp")

    # Write .inter file
    inter_df = ratings[["user_id", "item_id", "rating", "timestamp"]].copy()
    inter_df.columns = ["user_id:token", "item_id:token", "rating:float", "timestamp:float"]

    inter_df.to_csv(paths["inter_file"], sep="\t", index=False)
    print(f"Saved: {paths['inter_file']} ({len(inter_df):,} rows)")

    # Save ID mappings (store as str keys for JSON)
    user_map_save = {str(k): v for k, v in user_id_map.items()}
    item_map_save = {str(k): v for k, v in item_id_map.items()}
    paths["user_id_map"].write_text(json.dumps(user_map_save), encoding="utf-8")
    paths["item_id_map"].write_text(json.dumps(item_map_save), encoding="utf-8")
    print(f"Saved ID maps: {paths['user_id_map'].name}, {paths['item_id_map'].name}")

    return user_id_map, item_id_map

In [7]:
# RECBOLE CONFIG
def build_recbole_config(paths: dict) -> dict:
    """Build RecBole parameter dict for XSimGCL training."""
    config = {
        # Model
        "model": "XSimGCL",
        "dataset": "cinematch",
        "data_path": str(paths["recbole_data"].parent.resolve()),

        # Model hyperparams
        "embedding_size": EMBEDDING_SIZE,
        "n_layers": N_LAYERS,
        "cl_rate": CL_RATE,
        "noise_eps": NOISE_EPS,
        "reg_weight": REG_WEIGHT,

        # Data
        "USER_ID_FIELD": "user_id",
        "ITEM_ID_FIELD": "item_id",
        "RATING_FIELD": "rating",
        "TIME_FIELD": "timestamp",
        "load_col": {
            "inter": ["user_id", "item_id", "rating", "timestamp"],
        },
        "threshold": {"rating": RATING_THRESHOLD},

        # Eval
        "eval_args": {
            "split":    {"RS": SPLIT_RATIO},
            "group_by": "user",
            "order":    "TO",
            "mode":     "uni100",
        },
        "metrics":      ["Recall", "NDCG", "MRR"],
        "topk":         [10, 20, 50],
        "valid_metric": "NDCG@20",

        # Training
        "learning_rate": LEARNING_RATE,
        "train_batch_size": TRAIN_BATCH,
        "eval_batch_size": EVAL_BATCH,
        "epochs": EPOCHS,
        "stopping_step": EARLY_STOP,
        "weight_decay": 0.0,

        # GPU
        "gpu_id": 0 if torch.cuda.is_available() else -1,
        "use_gpu": torch.cuda.is_available(),
        "enable_amp": True,
        "enable_scaler": True,  
        "mixed_precision": True,


        # Misc
        "seed": 42,
        "reproducibility": False,
        "checkpoint_dir": str(paths["model_dir"] / "checkpoints"),
        "show_progress": True,
        "log_wandb": False,
    }

    import yaml
    with open(paths["config_yaml"], "w") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
    print(f"  Config saved: {paths['config_yaml']}")

    return config

In [8]:
def train_xsimgcl(paths: dict, config: dict):
    """Train XSimGCL via RecBole-GNN and export embeddings."""
    import torch
    from tqdm.auto import tqdm
    import tqdm as tqdm_module
    tqdm_module.tqdm = tqdm       
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("Training XSimGCL (RecBole-GNN)")
    print(f"{'─'*60}")

    try:
        from recbole_gnn.quick_start import run_recbole_gnn
        from recbole.utils import get_model, init_logger
        from recbole.config import Config
        from recbole.data import create_dataset, data_preparation
    except ImportError as e:
        print("\nRecBole-GNN not found or import failed.")
        raise

    # Train
    t0 = time.time()
    result = run_recbole_gnn(
        model="XSimGCL",
        dataset="cinematch",
        config_file_list=[str(paths["config_yaml"])], 
        config_dict=config,
    )

    train_time = time.time() - t0
    print(f"\n  Training completed in {train_time:.1f}s")
    print(f"  Best valid metric: {result.get('best_valid_score', 'N/A')}")
    print(f"  Test results: {result.get('test_result', 'N/A')}")

    return result, train_time

In [9]:
def export_embeddings(paths: dict, config: dict, result: dict):
    """Load the best checkpoint and export user/item embedding matrices."""
    print("  Exporting embeddings from best checkpoint")
    print(f"{'─'*60}")

    from recbole.config import Config
    from recbole_gnn.utils import create_dataset, data_preparation
    import recbole.config.configurator as configurator
    import recbole.utils.utils as rb_utils
    from recbole_gnn.model.general_recommender.xsimgcl import XSimGCL


    _original_get_model = rb_utils.get_model
    def _mock_get_model(model_name):
        if model_name.lower() == 'xsimgcl':
            return XSimGCL
        return _original_get_model(model_name)

    rb_utils.get_model = _mock_get_model
    configurator.get_model = _mock_get_model

    # Use the best checkpoint saved by RecBole
    checkpoint_path = result.get("best_valid_checkpoint")
    if not checkpoint_path:
        ckpt_dir = Path(config["checkpoint_dir"])
        checkpoints = sorted(ckpt_dir.glob("*.pth"), key=lambda p: p.stat().st_mtime)
        checkpoint_path = checkpoints[-1] if checkpoints else None

    assert checkpoint_path, "No checkpoint found"
    checkpoint_path = Path(checkpoint_path)
    print(f"  Checkpoint: {checkpoint_path}")

    # Re-init config + dataset (returning a Graph dataset)
    rb_config = Config(model="XSimGCL", dataset="cinematch", config_dict=config)
    dataset = create_dataset(rb_config)
    _, _, test_data = data_preparation(rb_config, dataset)

    # Load model
    model = XSimGCL(rb_config, dataset)
    checkpoint = torch.load(checkpoint_path, map_location="cpu") 
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    print(f"  Users: {dataset.user_num:,}  |  Items: {dataset.item_num:,}")

    # Extract embeddings (XSimGCL stores them in user/item_embedding)
    with torch.no_grad():
        user_emb = model.user_embedding.weight.data.cpu().numpy()
        item_emb = model.item_embedding.weight.data.cpu().numpy()

    print(f"  user_emb: {user_emb.shape}  |  item_emb: {item_emb.shape}")

    np.save(paths["user_emb"], user_emb)
    np.save(paths["item_emb"], item_emb)
    print(f"  Saved: {paths['user_emb'].name} and {paths['item_emb'].name}")

    return user_emb, item_emb

In [10]:
# DEMOGRAPHIC CLUSTERS

def build_demographic_clusters(
    user_emb_path: str | Path,
    user_id_map_path: str | Path,
    demo_profiles_path: str | Path,
    output_clusters_path: str | Path,
    output_map_path: str | Path,
) -> dict[str, np.ndarray]:
    """
    Build demographic cluster centroids from website users who have trained embeddings.

    Reads a CSV of website user demographics (columns: user_id, age_group, gender, region)
    and groups users by their demographic triple. For each group with >= MIN_CLUSTER_SIZE
    members, the centroid (mean embedding) is computed.

    Args:
        user_emb_path:        Path to user_embeddings.npy from training.
        user_id_map_path:     Path to user_id_map.json.
        demo_profiles_path:   Path to CSV with columns [user_id, age_group, gender, region].
        output_clusters_path: Where to save the cluster centroid matrix (.npy).
        output_map_path:      Where to save the cluster key mapping (.json).

    Returns:
        Dict mapping cluster key string → centroid vector.
    """
    print("Building demographic cluster centroids")
    print(f"{'─'*60}")

    user_emb = np.load(user_emb_path)
    with open(user_id_map_path, "r") as f:
        user_id_map = json.load(f)

    demos = pd.read_csv(demo_profiles_path, dtype=str)
    required_cols = {"user_id", "age_group", "gender", "region"}
    assert required_cols.issubset(demos.columns), (
        f"Demographics CSV must have columns {required_cols}, got {set(demos.columns)}"
    )

    # Collect embeddings per demographic group
    from collections import defaultdict
    cluster_vecs: dict[str, list[np.ndarray]] = defaultdict(list)

    matched, skipped = 0, 0
    for _, row in demos.iterrows():
        uid_str = str(row["user_id"])
        idx = user_id_map.get(uid_str)
        if idx is None or idx >= len(user_emb):
            skipped += 1
            continue
        matched += 1
        emb = user_emb[idx]
        profile = UserProfile(
            age_group=row.get("age_group", "undisclosed"),
            gender=row.get("gender", "undisclosed"),
            region=row.get("region", "Other"),
        )
        for key in profile.cluster_keys():
            cluster_vecs[str(key)].append(emb)

    print(f"  Matched {matched:,} users to embeddings (skipped {skipped:,})")

    # Compute centroids for clusters above minimum size
    centroids: dict[str, np.ndarray] = {}
    for key_str, vecs in cluster_vecs.items():
        if len(vecs) >= MIN_CLUSTER_SIZE:
            centroids[key_str] = np.mean(vecs, axis=0).astype(np.float32)

    print(f"  Clusters formed: {len(centroids)} (min size = {MIN_CLUSTER_SIZE})")

    # Save
    if centroids:
        keys = list(centroids.keys())
        matrix = np.stack([centroids[k] for k in keys])
        np.save(output_clusters_path, matrix)
        Path(output_map_path).write_text(json.dumps(keys), encoding="utf-8")
        print(f"  Saved: {Path(output_clusters_path).name} ({matrix.shape})")
        print(f"  Saved: {Path(output_map_path).name}")
    else:
        print(" No clusters met minimum size; skipping save.")

    return centroids

In [11]:

def _lookup_demographic_centroid(
    profile: UserProfile,
    demo_clusters_path: str | Path,
    demo_cluster_map_path: str | Path,
) -> np.ndarray | None:
    """
    Look up the best matching demographic centroid for a user profile.
    Tries exact match first, then progressively less specific keys.

    Returns None if no matching cluster exists.
    """
    cluster_map_path = Path(demo_cluster_map_path)
    clusters_path = Path(demo_clusters_path)

    if not cluster_map_path.exists() or not clusters_path.exists():
        return None

    with open(cluster_map_path, "r") as f:
        cluster_keys = json.load(f)  # list of key strings
    cluster_matrix = np.load(clusters_path)

    key_to_idx = {k: i for i, k in enumerate(cluster_keys)}

    for key in profile.cluster_keys():
        key_str = str(key)
        if key_str in key_to_idx:
            return cluster_matrix[key_to_idx[key_str]]

    return None

In [12]:

# NEW USER SUPPORT

def register_new_user(
    user_interactions: list[int],
    item_emb_path: str | Path,
    item_id_map_path: str | Path,
    user_profile: UserProfile | None = None,
    demo_clusters_path: str | Path | None = None,
    demo_cluster_map_path: str | Path | None = None,
) -> np.ndarray:
    """
    Generate an embedding for a new website user based on their interactions
    and (optionally) their demographic profile.

    Warm-start strategy:
      - Has interactions → mean of interacted item embeddings.
      - Has interactions + profile → blend: (1-α)·interaction_emb + α·demographic_centroid
        where α = DEMO_BLEND_WEIGHT, applied only when n_interactions < 5.
      - No interactions + profile → demographic cluster centroid.
      - No interactions, no profile → zero vector (gating sets β≈0).

    Args:
        user_interactions:      List of MovieLens movieIds the user has liked.
        item_emb_path:          Path to item_embeddings.npy
        item_id_map_path:       Path to item_id_map.json
        user_profile:           Optional UserProfile with age, gender, region.
        demo_clusters_path:     Optional path to demographic_clusters.npy
        demo_cluster_map_path:  Optional path to demographic_cluster_map.json

    Returns:
        User embedding vector of shape (embedding_size,)
    """
    item_emb = np.load(item_emb_path)
    with open(item_id_map_path, "r") as f:
        item_id_map = json.load(f)

    # Collect item embeddings for interactions
    vecs = []
    for movie_id in user_interactions:
        idx = item_id_map.get(str(movie_id))
        if idx is not None and idx < len(item_emb):
            vecs.append(item_emb[idx])

    # Try to get demographic centroid
    demo_centroid = None
    if user_profile and demo_clusters_path and demo_cluster_map_path:
        demo_centroid = _lookup_demographic_centroid(
            user_profile, demo_clusters_path, demo_cluster_map_path
        )

    if not vecs:
        # Cold start
        if demo_centroid is not None:
            return demo_centroid
        return np.zeros(item_emb.shape[1], dtype=np.float32)

    interaction_emb = np.mean(vecs, axis=0).astype(np.float32)

    # Blend with demographic centroid for users with few interactions
    if demo_centroid is not None and len(vecs) < 5:
        alpha = DEMO_BLEND_WEIGHT * (1.0 - len(vecs) / 5.0)  # decay as interactions grow
        interaction_emb = ((1 - alpha) * interaction_emb + alpha * demo_centroid).astype(np.float32)

    return interaction_emb

In [13]:
def register_batch_new_users(
    user_interaction_map: dict[str, list[int]],
    item_emb_path: str | Path,
    item_id_map_path: str | Path,
    output_path: str | Path | None = None,
    user_profiles: dict[str, UserProfile] | None = None,
    demo_clusters_path: str | Path | None = None,
    demo_cluster_map_path: str | Path | None = None,
) -> dict[str, np.ndarray]:
    """
    Batch register multiple new website users.

    Args:
        user_interaction_map:   {user_id: [movieId, ...]} for each new user
        item_emb_path:          Path to item_embeddings.npy
        item_id_map_path:       Path to item_id_map.json
        output_path:            If provided, save new user embeddings to this .npy file
        user_profiles:          Optional {user_id: UserProfile} for demographic warm-start
        demo_clusters_path:     Optional path to demographic_clusters.npy
        demo_cluster_map_path:  Optional path to demographic_cluster_map.json

    Returns:
        Dict mapping user_id to embedding vector
    """
    results = {}
    for uid, interactions in user_interaction_map.items():
        profile = user_profiles.get(uid) if user_profiles else None
        results[uid] = register_new_user(
            user_interactions=interactions,
            item_emb_path=item_emb_path,
            item_id_map_path=item_id_map_path,
            user_profile=profile,
            demo_clusters_path=demo_clusters_path,
            demo_cluster_map_path=demo_cluster_map_path,
        )

    if output_path:
        # Save as a matrix with a companion mapping
        user_ids = list(results.keys())
        user_matrix = np.stack([results[uid] for uid in user_ids])
        np.save(output_path, user_matrix)
        # Save ID list alongside
        map_path = Path(output_path).with_suffix(".json")
        map_path.write_text(json.dumps(user_ids), encoding="utf-8")
        print(f"New user embeddings: {user_matrix.shape} to {output_path}")

    return results

In [14]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [15]:
# MAIN
import sys
sys.path.insert(0, "/blue/egn6933/nagabhairava.r/RecBole-GNN")

def main():
    import time
    import pandas as pd
    import json
    t_start = time.time()
    paths = detect_paths()

    import numpy as np
    if not hasattr(np, 'float_'):
        np.float_ = np.float64
    if not hasattr(np, 'bool_'):
        np.bool_ = bool
    if not hasattr(np, 'int_'):
        np.int_ = np.int64
    if not hasattr(np, 'complex_'):
        np.complex_ = complex
    if not hasattr(np, 'object_'):
        np.object_ = object
    if not hasattr(np, 'unicode_'):
        np.unicode_ = np.str_

    user_id_map, item_id_map = convert_movielens_to_recbole(paths)
    print("  Building RecBole config")
    print(f"{'─'*60}")
    config = build_recbole_config(paths)


    print(f"  XSimGCL params: emb={EMBEDDING_SIZE}, layers={N_LAYERS}, "
          f"cl_rate={CL_RATE}, noise_eps={NOISE_EPS}")

    # Train
    result, train_time = train_xsimgcl(paths, config)

    
    # Export embeddings
    user_emb, item_emb = export_embeddings(paths, config, result)

    # Save manifest
    manifest = {
        "model": "xsimgcl",
        "framework": "RecBole",
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "dataset": {
            "source": str(paths["ratings_csv"]),
            "users": len(user_id_map),
            "items": len(item_id_map),
            "rating_threshold": RATING_THRESHOLD,
            "temporal_split": TEMPORAL_SPLIT,
        },
        "hyperparameters": {
            "embedding_size": EMBEDDING_SIZE,
            "n_layers": N_LAYERS,
            "cl_rate": CL_RATE,
            "noise_eps": NOISE_EPS,
            "reg_weight": REG_WEIGHT,
            "learning_rate": LEARNING_RATE,
            "epochs": EPOCHS,
            "early_stop": EARLY_STOP,
        },
        "results": {
            "best_valid_score": str(result.get("best_valid_score", "N/A")),
            "test_result": str(result.get("test_result", "N/A")),
            "train_time_seconds": round(train_time, 2),
        },
        "outputs": {
            "user_embedding_shape": list(user_emb.shape) if user_emb is not None else None,
            "item_embedding_shape": list(item_emb.shape) if item_emb is not None else None,
        },
    }
    paths["train_manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"\n Manifest saved: {paths['train_manifest']}")

    total = time.time() - t_start
    print(f"  XSimGCL DONE — Total time: {total:.1f}s")

if __name__ == "__main__":
    main()


Runtime: HPC
Converting MovieLens to RecBole .inter format
────────────────────────────────────────────────────────────
  Loaded 32,000,204 ratings
  After threshold (3.5): 20,228,336 positive interactions
  Users: 200,808  |  Items: 65,032
  Density: 0.1549%
Saved: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/dataset/cinematch/cinematch.inter (20,228,336 rows)
Saved ID maps: user_id_map.json, item_id_map.json
  Building RecBole config
────────────────────────────────────────────────────────────
  Config saved: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/xsimgcl_config.yaml
  XSimGCL params: emb=512, layers=3, cl_rate=0.5, noise_eps=0.1
Training XSimGCL (RecBole-GNN)
────────────────────────────────────────────────────────────


21 Mar 14:18    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 42
state = INFO
reproducibility = False
data_path = /blue/egn6933/nagabhairava.r/outputs/xsimgcl/dataset/cinematch
checkpoint_dir = /blue/egn6933/nagabhairava.r/outputs/xsimgcl/checkpoints
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 50
train_batch_size = 294912
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 5
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'RS': [0.8, 0.1, 0.1]}, 'group_by': 'user', 'order': 'TO', 'mode': 'uni100'}
repeatable = False
metrics = ['Recall', 'NDCG', 'MRR']
topk = [10, 20, 50]
valid_metric = NDCG@20
valid_metric_bigger = True
eval_batch_size

Train     0:   0%|                                                           | 0/56 [00:00<?, ?it/s…

21 Mar 14:21    INFO  epoch 0 training [time: 101.21s, train_loss1: 36.4221, train_loss2: 0.0005, train_loss3: 79.9292]


Evaluate   :   0%|                                                           | 0/41 [00:00<?, ?it/s…

...

Evaluate   :   0%|                                                           | 0/41 [00:00<?, ?it/s…

21 Mar 16:49    INFO  best valid : OrderedDict({'recall@10': 0.7505, 'recall@20': 0.8496, 'recall@50': 0.9308, 'ndcg@10': 0.7647, 'ndcg@20': 0.7889, 'ndcg@50': 0.819, 'mrr@10': 0.8301, 'mrr@20': 0.8305, 'mrr@50': 0.8305})
21 Mar 16:49    INFO  test result: OrderedDict({'recall@10': 0.7268, 'recall@20': 0.8293, 'recall@50': 0.9163, 'ndcg@10': 0.7292, 'ndcg@20': 0.7558, 'ndcg@50': 0.7888, 'mrr@10': 0.8041, 'mrr@20': 0.8046, 'mrr@50': 0.8047})



  Training completed in 9101.6s
  Best valid metric: 0.7889
  Test results: OrderedDict({'recall@10': 0.7268, 'recall@20': 0.8293, 'recall@50': 0.9163, 'ndcg@10': 0.7292, 'ndcg@20': 0.7558, 'ndcg@50': 0.7888, 'mrr@10': 0.8041, 'mrr@20': 0.8046, 'mrr@50': 0.8047})
  Exporting embeddings from best checkpoint
────────────────────────────────────────────────────────────
  Checkpoint: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/checkpoints/XSimGCL-Mar-21-2026_14-19-36.pth


21 Mar 16:51    INFO  [Training]: train_batch_size = [294912] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
21 Mar 16:51    INFO  [Evaluation]: eval_batch_size = [40000000] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'group_by': 'user', 'order': 'TO', 'mode': 'uni100'}]


  Users: 200,809  |  Items: 65,033
  user_emb: (200809, 512)  |  item_emb: (65033, 512)
  Saved: user_embeddings.npy and item_embeddings.npy

 Manifest saved: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/train_manifest.json
  XSimGCL DONE — Total time: 9216.5s


In [16]:
# torch.cuda.empty_cache()